In [13]:
# 6_feature_engineering.py
def create_player_features(player_id):
    """Create features for a player"""
    
    # Load data
    players = pd.read_csv('Data/current_players.csv')
    history = pd.read_csv('Data/player_history.csv')
    
    # Get player info
    player = players[players['id'] == player_id].iloc[0]
    player_history = history[history['player_id'] == player_id].sort_values('round')
    
    features = {}
    
    # Recent form (last 5 games)
    if len(player_history) >= 5:
        recent_5 = player_history.tail(5)
        features['avg_points_last_5'] = recent_5['total_points'].mean()
        features['avg_minutes_last_5'] = recent_5['minutes'].mean()
        features['goals_last_5'] = recent_5['goals_scored'].sum()
        features['assists_last_5'] = recent_5['assists'].sum()
    else:
        features['avg_points_last_5'] = 0
        features['avg_minutes_last_5'] = 0
        features['goals_last_5'] = 0
        features['assists_last_5'] = 0
    
    # Season stats
    features['season_points'] = player['total_points']
    features['season_minutes'] = player['minutes']
    features['points_per_game'] = player['points_per_game']
    
    # Player info
    features['position'] = player['element_type']
    features['team'] = player['team']
    features['price'] = player['price']
    features['selected_by_percent'] = player['selected_by_percent']
    
    return features

# Create training dataset
def prepare_ml_data():
    history = pd.read_csv('Data/player_history.csv')
    
    X = []
    y = []
    
    # Get unique player-gameweek combinations
    for player_id in history['player_id'].unique()[:100]:  # Start with 100 players
        player_hist = history[history['player_id'] == player_id].sort_values('round')
        
        if len(player_hist) > 5:
            for i in range(5, len(player_hist)):
                # Features from games before this one
                features = create_player_features(player_id)
                X.append(features)
                
                # Target: points in this game
                y.append(player_hist.iloc[i]['total_points'])
    
    X_df = pd.DataFrame(X)
    y_series = pd.Series(y)
    
    # Save prepared data
    X_df.to_csv('Data/features.csv', index=False)
    y_series.to_csv('Data/targets.csv', index=False)
    
    print(f"✅ Created {len(X)} training samples")
    return X_df, y_series

X, y = prepare_ml_data()

✅ Created 679 training samples


In [15]:
# 7_train_model.py
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# Load prepared data
X = pd.read_csv('Data/features.csv')
y = pd.read_csv('Data/targets.csv').squeeze()

# Fill missing values
X = X.fillna(0)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train models
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"RF MAE: {rf_mae:.2f}")

print("\nTraining XGBoost...")
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f"XGB MAE: {xgb_mae:.2f}")

# Save best model
best_model = xgb_model if xgb_mae < rf_mae else rf_model
joblib.dump(best_model, 'models/predictor.pkl')
print(f"\n✅ Saved best model (MAE: {min(rf_mae, xgb_mae):.2f})")

# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top Features:")
print(importance.head(10))

Training Random Forest...
RF MAE: 1.30

Training XGBoost...
XGB MAE: 1.32

✅ Saved best model (MAE: 1.30)

📊 Top Features:
                feature  importance
0     avg_points_last_5    0.395454
4         season_points    0.232217
1    avg_minutes_last_5    0.223607
6       points_per_game    0.045879
5        season_minutes    0.032243
10  selected_by_percent    0.029085
9                 price    0.017598
3        assists_last_5    0.007915
2          goals_last_5    0.007815
7              position    0.006410


In [18]:
# IMPROVED PREDICTION MODEL - considering fixtures and form
import pandas as pd
import numpy as np

def get_fixture_difficulty(team_id, gameweek):
    """Get fixture difficulty for a team in a gameweek"""
    # In reality, you'd fetch this from FPL API
    # For now, return random 1-5 (1=easy, 5=hard)
    return np.random.randint(1, 6)

def predict_next_gameweek_advanced():
    """Enhanced predictions considering form, fixtures, and opponent"""
    
    players = pd.read_csv('Data/current_players.csv')
    history = pd.read_csv('Data/player_history.csv')
    model = joblib.load('models/predictor.pkl')
    
    # Add position names
    position_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
    players['position'] = players['element_type'].map(position_map)
    
    predictions = []
    
    for _, player in players.iterrows():
        try:
            # Base features (as before)
            player_hist = history[history['player_id'] == player['id']]
            
            features = {}
            
            # Recent form (weighted - recent games matter more)
            if len(player_hist) >= 3:
                recent = player_hist.tail(3)
                features['form_score'] = recent['total_points'].mean() * 1.2  # 20% boost for form
            else:
                features['form_score'] = player['points_per_game']
            
            # Fixture difficulty (would need real fixture data)
            features['fixture_difficulty'] = 3  # Placeholder - medium difficulty
            
            # Home/Away boost (home games typically score more)
            features['is_home'] = 1  # Placeholder
            
            # Position-specific scoring patterns
            position_multipliers = {
                'GK': 0.8,   # GKs score less
                'DEF': 1.0,  # Defenders baseline
                'MID': 1.2,  # Mids score more
                'FWD': 1.3   # Forwards highest variance
            }
            
            base_prediction = features['form_score']
            
            # Apply position multiplier
            position = position_map.get(player['element_type'], 'MID')
            base_prediction *= position_multipliers.get(position, 1.0)
            
            # Fixture adjustment (easy fixtures boost score)
            fixture_adj = (6 - features['fixture_difficulty']) / 5  # Convert to multiplier
            base_prediction *= (0.8 + 0.4 * fixture_adj)  # 0.8 to 1.2x based on fixture
            
            # Captain potential (top players get slight boost)
            if player['total_points'] > 100:  # High scorers this season
                base_prediction *= 1.1
            
            predictions.append({
                'id': player['id'],
                'name': player['web_name'],
                'team': player.get('name_team', 'Unknown'),
                'position': position,
                'price': player['now_cost'] / 10,
                'predicted_points': round(max(2, min(15, base_prediction)), 1),  # Cap between 2-15
                'form': player_hist.tail(5)['total_points'].mean() if len(player_hist) >= 5 else 0,
                'ownership': float(player['selected_by_percent'])
            })
            
        except:
            continue
    
    predictions_df = pd.DataFrame(predictions)
    
    # Adjust for gameweek - some weeks score higher (double gameweeks, etc.)
    # This would use real fixture data in production
    gameweek_multiplier = 1.0  # Could be 2.0 for double gameweek
    predictions_df['predicted_points'] *= gameweek_multiplier
    
    predictions_df.to_csv('Data/predictions_advanced.csv', index=False)
    
    print(f"✅ Made predictions for {len(predictions)} players")
    print("\n📊 NEXT GAMEWEEK PREDICTIONS:")
    print("="*60)
    
    # Show top captain choices
    print("\n🏅 TOP CAPTAIN CHOICES (2x points):")
    top_captains = predictions_df.nlargest(5, 'predicted_points')
    for _, p in top_captains.iterrows():
        captain_points = p['predicted_points'] * 2
        print(f"  {p['name']:15} {p['position']:3} £{p['price']:.1f}M  "
              f"{p['predicted_points']:.1f}pts (C: {captain_points:.1f}pts)")
    
    # Show differentials (low ownership, high predicted points)
    print("\n💎 DIFFERENTIALS (low ownership, high upside):")
    differentials = predictions_df[predictions_df['ownership'] < 10].nlargest(5, 'predicted_points')
    for _, p in differentials.iterrows():
        print(f"  {p['name']:15} {p['position']:3} {p['ownership']:.1f}% owned  "
              f"{p['predicted_points']:.1f}pts")
    
    # Show by position
    for pos in ['GK', 'DEF', 'MID', 'FWD']:
        print(f"\n{pos} Predictions:")
        pos_players = predictions_df[predictions_df['position'] == pos].nlargest(3, 'predicted_points')
        for _, p in pos_players.iterrows():
            print(f"  {p['name']:15} £{p['price']:.1f}M  {p['predicted_points']:.1f}pts")
    
    return predictions_df

# Run improved predictions
predictions = predict_next_gameweek_advanced()

✅ Made predictions for 755 players

📊 NEXT GAMEWEEK PREDICTIONS:

🏅 TOP CAPTAIN CHOICES (2x points):
  Trossard        MID £6.9M  13.5pts (C: 27.0pts)
  Gibbs-White     MID £7.3M  13.0pts (C: 26.0pts)
  Eze             MID £7.7M  12.5pts (C: 25.0pts)
  Barnes          MID £6.3M  12.5pts (C: 25.0pts)
  Dewsbury-Hall   MID £4.9M  12.0pts (C: 24.0pts)

💎 DIFFERENTIALS (low ownership, high upside):
  Trossard        MID 1.1% owned  13.5pts
  Gibbs-White     MID 3.9% owned  13.0pts
  Barnes          MID 1.0% owned  12.5pts
  Dewsbury-Hall   MID 2.9% owned  12.0pts
  Keane           DEF 1.7% owned  11.6pts

GK Predictions:
  Henderson       £5.0M  7.0pts
  Verbruggen      £4.4M  6.3pts
  Pickford        £5.5M  6.3pts

DEF Predictions:
  Keane           £4.5M  11.6pts
  Muñoz           £5.9M  10.8pts
  Lacroix         £5.0M  10.4pts

MID Predictions:
  Trossard        £6.9M  13.5pts
  Gibbs-White     £7.3M  13.0pts
  Eze             £7.7M  12.5pts

FWD Predictions:
  Wilson          £5.8M  11